##### RAG pipeline using Hugging Face embeddings, FAISS, and LangChain, covering both text and PDF workflows

In [ ]:
import os

In [ ]:
from huggingface_hub import login

login()

In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:

res = embeddings.embed_query("hello AI")

In [ ]:
len(res)

### Cosine Similarity

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

documents=["what is a capital of USA?",
           "Who is a president of USA?",
           "Who is a prime minister of India?"]


In [ ]:

my_query="Narendra modi is prime minister of india?"

In [ ]:

document_embedding=embeddings.embed_documents(documents)

In [ ]:

len(document_embedding)

In [ ]:

query_embedding=embeddings.embed_query(my_query)

In [ ]:

len(query_embedding)

In [ ]:

cosine_similarity([query_embedding],document_embedding)

| Index | Similarity | Interpretation                   |
| ----- | ---------- | -------------------------------- |
| 0     | **0.1176** | Very weak similarity             |
| 1     | **0.3432** | Low to moderate similarity       |
| 2     | **0.8141** | **High similarity (best match)** |


| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


### FIASS Vector database

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import faiss

from langchain_community.vectorstores import FAISS

from langchain_community.docstore.in_memory import InMemoryDocstore


<b>InMemoryDocstore = stores documents temporarily in your computer's memory (RAM).
</b>

| Part        | Meaning                                                        |
| ----------- | -------------------------------------------------------------- |
| `IndexFlat` | Flat (brute-force) index – compares query with **all vectors** |
| `L2`        | Uses **Euclidean (L2) distance**                               |
| `384`       | Embedding vector dimension                                     |


With L2 distance:

    Smaller value = more similar
    
    0 = identical vectors
    
    Larger distance = less relevant

In [ ]:

index=faiss.IndexFlatL2(384)

#### cosine similarity

#### index = faiss.IndexFlatIP(384)  # Inner Product

| Index         | Use when                                    |
| ------------- | ------------------------------------------- |
| `IndexFlatL2` | Raw embeddings, Euclidean space             |
| `IndexFlatIP` | **Cosine similarity** (after normalization) |


In [ ]:

index

In [ ]:

vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

| Part                            | Simple meaning                            |
| ------------------------------- | ----------------------------------------- |
| `FAISS(...)`                    | Creates a FAISS vector database           |
| `embedding_function=embeddings` | Converts text into vectors                |
| `index=index`                   | Stores/searches the vectors               |
| `docstore=InMemoryDocstore()`   | Stores the original documents in RAM      |
| `index_to_docstore_id={}`       | Maps FAISS vector numbers to document IDs |


In [ ]:

vector_store.add_texts(["AI is future","AI is powerful","Dogs are cute"])

In [ ]:

vector_store.index_to_docstore_id

In [ ]:

results = vector_store.similarity_search("Tell me about AI", k=2)

In [ ]:

results

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


### Working with Text docs

In [57]:
# from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [58]:
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [59]:

index=faiss.IndexFlatIP(384)

vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)


In [60]:

vector_store.add_documents(documents=documents)

['a6b9ffb9-b54a-4943-942b-290549d180af',
 '6c1156fc-9740-4271-a85d-8339157f3ae5',
 '3e263c92-290e-4b6d-a043-9bf132d25b2f',
 '635e4686-d3e0-4d50-aced-b2ba7af29da7',
 'b5e219ca-5ede-4576-86c0-0c66c464f817',
 'c491cb75-51ec-43e5-8890-0c35f5ba5a26',
 '60d7ab25-d932-4673-8da9-5fa70e30139b',
 '9e0d4021-218e-49a8-8a46-77fd58c90b77',
 '2ac28dd3-8736-4068-b658-47b4ee8d4558',
 '3ef53998-a785-466a-840c-3389119a25d9']

In [61]:

vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2
)

[Document(id='3e263c92-290e-4b6d-a043-9bf132d25b2f', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9e0d4021-218e-49a8-8a46-77fd58c90b77', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]


The query text is converted into an embedding

FAISS compares it with all stored document embeddings

It ranks them by similarity (L2 distance or cosine, depending on index)

Returns the top 2 most similar documents

---

In [62]:

vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2, #hyperparameter,
    filter={"source":{"$eq": "tweet"}}
)


[Document(id='3e263c92-290e-4b6d-a043-9bf132d25b2f', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9e0d4021-218e-49a8-8a46-77fd58c90b77', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]


“Retrieve documents semantically similar to the query only from documents whose metadata source == 'tweet'.”


---

In [63]:

result=vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2, #hyperparameter,
    filter={"source":"news"}
)



“Return only documents that are semantically similar AND whose metadata has

source = "news".”


In [64]:

result[0].metadata

{'source': 'news'}

In [65]:

result[0].page_content

'Robbers broke into the city bank and stole $1 million in cash.'

In [67]:

retriever=vector_store.as_retriever(search_kwargs={"k": 2})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E389C919A0>, search_kwargs={'k': 2})

In [68]:

retriever.invoke("LangChain provides abstractions to make working with LLMs easy")


[Document(id='3e263c92-290e-4b6d-a043-9bf132d25b2f', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9e0d4021-218e-49a8-8a46-77fd58c90b77', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [69]:

vector_store.save_local("vcube_faiss index")

In [70]:

new_vector_store=FAISS.load_local(
  "vcube_faiss index",embeddings ,allow_dangerous_deserialization=True
)


In [71]:

new_vector_store.similarity_search("langchain")


[Document(id='3e263c92-290e-4b6d-a043-9bf132d25b2f', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9e0d4021-218e-49a8-8a46-77fd58c90b77', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='b5e219ca-5ede-4576-86c0-0c66c464f817', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='60d7ab25-d932-4673-8da9-5fa70e30139b', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]